# SHM Pipeline — Lattice Defect Classification
**Now:** healthy vs defective (missing strut)  
**Later:** defect localization

Data: Ansys harmonic FRF CSVs (`nodal_displacement_complex.csv`)

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
# Run once per session
!pip install -q scipy scikit-learn joblib pyuff

In [ ]:
# ── Mount Google Drive & set repo path ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
REPO = '/content/drive/MyDrive/PyMechanical-Single-Cell-Lattice'
sys.path.insert(0, f'{REPO}/ml')

DATA_DIR = '/content/drive/MyDrive/PyMechanical/Single Cell Lattice'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from features  import load_frf_pipeline, extract_features
from dataset   import build_dataset, save_dataset, load_dataset, LABEL_NAMES
from classifier import train, evaluate, predict

## 1 · Load & Visualise FRFs

In [ ]:
FORCE = dict(force_x_mm=10.0, force_y_mm=-10.0, force_z_mm=31.5)

healthy_csv  = f"{DATA_DIR}/nodal_displacement_complex.csv"
damaged_csv  = f"{DATA_DIR}/Missing Strut/damaged_displacement_complex.csv"

h_freqs, h_amp = load_frf_pipeline(healthy_csv,  **FORCE)
d_freqs, d_amp = load_frf_pipeline(damaged_csv,  **FORCE)

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(h_freqs, h_amp, label='Healthy',       color='steelblue')
ax.semilogy(d_freqs, d_amp, label='Missing strut', color='darkorange', linestyle='--')
ax.set_xlim(500, 3500)
ax.set_xlabel('Frequency (Hz)', fontsize=13)
ax.set_ylabel('Amplitude (mm, log)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 2 · Feature Extraction Demo

In [ ]:
h_feats = extract_features(h_freqs, h_amp, n_peaks=5, min_freq=500)
d_feats = extract_features(d_freqs, d_amp, n_peaks=5, min_freq=500)

comparison = pd.DataFrame([h_feats, d_feats], index=['healthy', 'damaged'])
print(comparison.to_string())

In [ ]:
# Visualise extracted peaks on the FRF
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, freqs, amp, feats, title, color in [
    (axes[0], h_freqs, h_amp, h_feats, 'Healthy',       'steelblue'),
    (axes[1], d_freqs, d_amp, d_feats, 'Missing strut', 'darkorange'),
]:
    ax.semilogy(freqs, amp, color=color, linewidth=1.2)
    for i in range(5):
        pf = feats.get(f'peak_freq_{i}')
        pa = feats.get(f'peak_amp_{i}')
        if pf and not np.isnan(pf):
            ax.axvline(pf, color=color, linewidth=0.8, alpha=0.6, linestyle=':')
            ax.plot(pf, pa, 'o', color=color, markersize=7)
    ax.set_xlim(500, 3500)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Frequency (Hz)', fontsize=12)
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_ylabel('Amplitude (mm, log)', fontsize=12)
plt.tight_layout()
plt.show()

## 3 · Build Dataset

Add new samples here as you collect more data. Each entry is one FRF CSV.

In [ ]:
SAMPLES = [
    {
        'csv_path' : healthy_csv,
        'label'    : 0,           # 0 = healthy
        'csv_type' : 'pipeline',
        **FORCE,
        'meta'     : {'geometry': 'single_cell_30mm', 'excitation': 'center'},
    },
    {
        'csv_path' : damaged_csv,
        'label'    : 1,           # 1 = defective
        'csv_type' : 'pipeline',
        **FORCE,
        'meta'     : {'geometry': 'single_cell_30mm_missing_strut', 'excitation': 'center'},
    },
    # ── Add new samples below ────────────────────────────────────────────────
    # {'csv_path': '...', 'label': 0, 'csv_type': 'pipeline', **FORCE, 'meta': {}},
]

X, y, meta = build_dataset(SAMPLES, n_peaks=5, min_freq=500)
print(f"Dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(X)

In [ ]:
# Save feature matrix to Drive for reuse
DATASET_PATH = f"{DATA_DIR}/shm_dataset.csv"
save_dataset(X, y, DATASET_PATH)

## 4 · Train & Evaluate

> **Note:** LOO cross-validation needs at least 2 samples *per class*.  
> With 1 healthy + 1 defective you can fit the model but not get a meaningful accuracy estimate.  
> Keep collecting data — aim for 10+ samples per class before reading too much into scores.

In [ ]:
# Load from saved CSV (skip build_dataset if already saved)
# X, y = load_dataset(DATASET_PATH)

if len(np.unique(y)) < 2 or np.min(np.bincount(y)) < 2:
    print("Not enough samples for cross-validation yet. Collect more data.")
    model = train(X, y, model='rf')   # fit anyway — useful for predict()
else:
    model = evaluate(X, y, model='rf')

## 5 · Predict a New Sample

In [ ]:
# Predict on a new CSV (e.g. experimental PSV-derived FRF)
from features import load_frf_simple

# new_freqs, new_amp = load_frf_simple('path/to/new_frf.csv')
# new_feats = extract_features(new_freqs, new_amp, n_peaks=5, min_freq=500)
# label, proba = predict(model, new_feats)
# print(f"Prediction: {LABEL_NAMES[label]}  (p={proba[label]:.2f})")

## 6 · Localization Roadmap (future)

When you have damage cases at multiple known locations:
- Change `label` from binary (0/1) to multi-class (0=healthy, 1=missing_strut_A, 2=missing_strut_B, …)
- Add `'defect_location': (x, y, z)` to each defective sample's `meta`
- `evaluate()` already prints per-class metrics — nothing else changes in the pipeline
- For regression (predict exact XYZ): swap classifier for a `MultiOutputRegressor` wrapping RF

For PyTorch (when N is large enough):
- Replace `features.py` with a 1-D CNN that takes the raw amplitude spectrum
- `dataset.py` becomes a `torch.utils.data.Dataset` wrapping the same CSVs
- Classifier head stays the same; swap LOO for stratified k-fold